# 面向噪声数据的图像识别系统

课程：《机器学习与 Python 编程》研究性专题（工业工程）

数据集：MNIST-C。本报告覆盖指导书中的必做部分和选做部分，实验采用 `identity`、`shot_noise`、`rotate` 作为训练数据来源，并在全部本地 MNIST-C 测试集上评估组合训练模型。

## 1. 研究背景

MNIST-C 是在 MNIST 图像上施加多种常见破坏后得到的鲁棒性基准。Zenodo 页面说明官方 `mnist_c.zip` 包含 15 个 corrupted 版本；本地目录还包含未破坏的 `identity`，因此本实验共评估 16 个集合。指导书中“16 组噪声 + identity = 17 组”的表述与官方数据页和本地目录不一致，这里以本地实际数据和官方说明为准。

参考资料：

- Zenodo: https://zenodo.org/records/3239543
- Google Research MNIST-C 源码: https://github.com/google-research/mnist-c
- Mu, N. and Gilmer, J. MNIST-C: A robustness benchmark for computer vision. arXiv:1906.02337.

## 2. 队伍分工说明

| 成员 | 分工 |
|---|---|
| 组长 | 任务拆解、实验方案设计、报告统稿、展示组织 |
| 成员 A | 数据读取、数据划分、预处理对比实验 |
| 成员 B | 神经网络模型训练、损失函数与参数搜索实验 |
| 成员 C | 选做鲁棒性实验、图表整理、结果分析 |

> 如需提交时写真实姓名，可直接把上表中的占位成员替换为本组成员。

## 3. 讨论记录

| 时间 | 讨论主题 | 结论 |
|---|---|---|
| 第 1 次 | 数据与任务边界 | 确认使用 `identity`、`shot_noise`、`rotate` 完成必做训练，并用本地全部 16 个数据集完成选做评估。 |
| 第 2 次 | 模型方案 | 选择 3 个神经网络方案：单隐层 MLP、双隐层 MLP、PCA+MLP。PCA 是无监督特征压缩，用于比较表示学习对分类的影响。 |
| 第 3 次 | 对比实验 | 数据处理比较验证集比例、标准化、PCA；损失函数比较交叉熵、标签平滑、MSE；参数搜索比较隐藏层、L2 正则和学习率。 |
| 第 4 次 | 结果解释 | 重点分析 identity 模型跨噪声性能下降，以及组合训练对不同破坏类型的鲁棒性。 |

## 4. 方案设计

### 4.1 数据处理

每张图像原始尺寸为 28 x 28 x 1，读取后展平成 784 维向量，像素缩放到 `[0, 1]`。训练集再用分层抽样划分训练/验证集，测试集使用官方 `test` split。为了保证实验可复现，所有随机过程固定 `random_state=42`。

对比的数据处理方案包括：仅缩放、标准化、标准化后 PCA 64 维，以及 80/20 和 90/10 两种训练/验证划分。

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
required = pd.read_csv(TABLE_DIR / 'required_results.csv')
preprocessing = pd.read_csv(TABLE_DIR / 'preprocessing_study.csv')
hyper = pd.read_csv(TABLE_DIR / 'hyperparameter_search.csv')
loss = pd.read_csv(TABLE_DIR / 'loss_comparison.csv')
kmeans = pd.read_csv(TABLE_DIR / 'kmeans_baseline.csv')
cv_summary = pd.read_csv(TABLE_DIR / 'cross_validation_summary.csv')
optional = pd.read_csv(TABLE_DIR / 'optional_all_datasets_accuracy.csv')

### 4.2 模型选择

必做部分对每个训练数据集分别训练 3 个神经网络：

| 模型 | 说明 | 作用 |
|---|---|---|
| `mlp_1hidden` | 标准化 + 64 单隐层 MLP | 基线神经网络 |
| `mlp_2hidden` | 标准化 + 128/64 双隐层 MLP | 增加非线性容量 |
| `pca_mlp` | 标准化 + PCA(64) + 单隐层 MLP | 使用无监督 PCA 表示后再分类 |

这 3 个方案覆盖了监督学习神经网络和无监督特征学习 + 监督分类的组合方式。此外，新增 `pca64_kmeans_majority_vote` 作为无监督 KMeans 基线：训练阶段只用图像聚类，评价阶段再用多数投票把簇映射到数字标签。

### 4.3 损失衡量与参数优化

scikit-learn 的 `MLPClassifier` 使用交叉熵进行多分类训练。为满足损失函数对比，本实验另外实现了一个两层 NumPy MLP，比较交叉熵、标签平滑交叉熵和 MSE。参数优化使用网格搜索思想，比较隐藏层规模、L2 正则系数和学习率对验证集准确率的影响。

## 5. 数据处理实验结果

| dataset   | preprocessing     |   validation_split |   train_samples |   val_samples |   val_accuracy |   val_macro_f1 |   fit_seconds |
|:----------|:------------------|-------------------:|----------------:|--------------:|---------------:|---------------:|--------------:|
| identity  | scale01           |                0.2 |            6400 |          1600 |        0.89875 |       0.897385 |      0.223284 |
| identity  | scale01           |                0.1 |            7200 |           800 |        0.90375 |       0.902805 |      0.263153 |
| identity  | standardize       |                0.2 |            6400 |          1600 |        0.90375 |       0.903082 |      0.284554 |
| identity  | standardize       |                0.1 |            7200 |           800 |        0.9125  |       0.912017 |      0.306727 |
| identity  | pca64_standardize |                0.2 |            6400 |          1600 |        0.83875 |       0.836249 |      0.240409 |

数据处理结果表明，标准化通常比只做 `[0, 1]` 缩放更适合 MLP，因为各像素维度的尺度被统一后，Adam 优化更稳定。PCA 能压缩噪声和维度，但也可能丢失部分笔画细节，因此它更适合作为鲁棒性和速度的折中方案。

## 6. 损失函数对比

| dataset   | network             | loss            |   label_smoothing |   train_samples |   val_samples |   val_accuracy |   val_macro_f1 |   fit_seconds |
|:----------|:--------------------|:----------------|------------------:|----------------:|--------------:|---------------:|---------------:|--------------:|
| identity  | numpy_two_layer_mlp | cross_entropy   |               0   |            6400 |          1600 |       0.9025   |       0.901578 |      0.259755 |
| identity  | numpy_two_layer_mlp | label_smoothing |               0.1 |            6400 |          1600 |       0.893125 |       0.892155 |      0.26076  |
| identity  | numpy_two_layer_mlp | mse             |               0   |            6400 |          1600 |       0.575625 |       0.538473 |      0.264511 |

损失函数实验中，验证集准确率最高的是 `cross_entropy`，准确率为 0.902。交叉熵直接优化类别概率，通常比 MSE 更适合多分类；标签平滑会牺牲一点训练集拟合能力，但能降低过度自信，对噪声标签或分布偏移更稳健。

## 7. 参数搜索结果

| dataset   | hidden_layer_sizes   |   alpha_l2 |   learning_rate_init |   train_samples |   val_samples |   val_accuracy |   val_macro_f1 |   fit_seconds |
|:----------|:---------------------|-----------:|---------------------:|----------------:|--------------:|---------------:|---------------:|--------------:|
| identity  | (64,)                |     0.0001 |                0.001 |            6400 |          1600 |       0.90375  |       0.903082 |      0.285746 |
| identity  | (128,)               |     0.0001 |                0.001 |            6400 |          1600 |       0.904375 |       0.903653 |      0.380717 |
| identity  | (128, 64)            |     0.0001 |                0.001 |            6400 |          1600 |       0.91375  |       0.912976 |      0.530776 |
| identity  | (128, 64)            |     0.001  |                0.001 |            6400 |          1600 |       0.91375  |       0.912954 |      0.491786 |
| identity  | (128, 64)            |     0.0001 |                0.003 |            6400 |          1600 |       0.9225   |       0.921591 |      0.527605 |
| identity  | (256, 128)           |     0.0001 |                0.001 |            6400 |          1600 |       0.928125 |       0.927451 |      0.95993  |

参数搜索中较优配置为隐藏层 `(256, 128)`、L2 正则 `0.0001`、学习率 `0.001`，验证集准确率为 0.928。模型容量过小容易欠拟合，容量过大或学习率过高则更容易在噪声图像上不稳定。

## 8. 无监督 KMeans 基线

为更明确地覆盖无监督学习，本实验增加 `PCA(64) + KMeans(10)`。KMeans 训练时不使用标签；训练完成后，仅为了评价聚类结果，用训练集多数投票将每个簇映射为一个数字类别。该方法通常低于监督神经网络，但可以反映 MNIST-C 数据在无标签条件下的自然可分性。

| experiment               | train_dataset   | eval_dataset   | model                      |   train_samples |   val_samples |   test_samples |   val_accuracy |   val_macro_f1 |   test_accuracy |   test_macro_f1 |   fit_seconds |
|:-------------------------|:----------------|:---------------|:---------------------------|----------------:|--------------:|---------------:|---------------:|---------------:|----------------:|----------------:|--------------:|
| kmeans_same_corruption   | identity        | identity       | pca64_kmeans_majority_vote |            6400 |          1600 |          10000 |       0.364375 |       0.246877 |          0.3717 |        0.25254  |      0.523496 |
| kmeans_identity_to_noise | identity        | shot_noise     | pca64_kmeans_majority_vote |            6400 |          1600 |          10000 |       0.364375 |       0.246877 |          0.3661 |        0.250827 |      0.523496 |
| kmeans_identity_to_noise | identity        | rotate         | pca64_kmeans_majority_vote |            6400 |          1600 |          10000 |       0.364375 |       0.246877 |          0.2988 |        0.203753 |      0.523496 |
| kmeans_same_corruption   | shot_noise      | shot_noise     | pca64_kmeans_majority_vote |            6400 |          1600 |          10000 |       0.29625  |       0.227484 |          0.2958 |        0.218624 |      0.460444 |
| kmeans_same_corruption   | rotate          | rotate         | pca64_kmeans_majority_vote |            6400 |          1600 |          10000 |       0.270625 |       0.184777 |          0.2693 |        0.183753 |      0.520038 |

![KMeans 无监督基线准确率](outputs/figures/kmeans_baseline_accuracy.png)

KMeans 基线中表现最好的记录是 `identity` -> `identity`，测试准确率为 0.372。它的准确率低于 MLP，原因是 KMeans 只按像素空间距离形成簇，不能直接学习数字类别边界。

## 9. 交叉验证

除单次训练/验证划分外，本实验在 `identity` 训练集上进行 3 折交叉验证。每一折轮流作为验证集，其余两折训练模型，最后比较平均准确率和标准差。这能减少单次划分带来的偶然性，更符合指导书中“不同验证方法”的要求。

| model       |   mean_val_accuracy |   std_val_accuracy |   mean_val_macro_f1 |   std_val_macro_f1 |   mean_fit_seconds |
|:------------|--------------------:|-------------------:|--------------------:|-------------------:|-------------------:|
| mlp_2hidden |            0.911167 |         0.00875119 |            0.910186 |         0.00933367 |           0.352543 |
| mlp_1hidden |            0.897    |         0.007      |            0.895711 |         0.00754113 |           0.196306 |
| pca_mlp     |            0.801    |         0.011      |            0.797261 |         0.0111953  |           0.190737 |

![3 折交叉验证准确率](outputs/figures/cross_validation_accuracy.png)

3 折交叉验证中平均验证准确率最高的是 `mlp_2hidden`，平均准确率为 0.911，标准差为 0.009。

## 10. 必做部分实验结果

下表包含两类结果：

- `required_same_corruption`：在 `identity`、`shot_noise`、`rotate` 上分别训练 3 个神经网络，并在同类测试集上评价。
- `identity_to_noise`：用 `identity` 训练的 3 个神经网络，分别测试到 `shot_noise` 和 `rotate`，用于观察分布外鲁棒性。

| experiment               | train_dataset   | eval_dataset   | model       |   train_samples |   val_samples |   test_samples |   val_accuracy |   val_macro_f1 |   test_accuracy |   test_macro_f1 |   fit_seconds |
|:-------------------------|:----------------|:---------------|:------------|----------------:|--------------:|---------------:|---------------:|---------------:|----------------:|----------------:|--------------:|
| required_same_corruption | identity        | identity       | mlp_1hidden |           12000 |          3000 |          10000 |       0.938    |       0.937405 |          0.9418 |        0.941146 |      0.841977 |
| identity_to_noise        | identity        | shot_noise     | mlp_1hidden |           12000 |          3000 |          10000 |       0.938    |       0.937405 |          0.9229 |        0.921918 |      0.841977 |
| identity_to_noise        | identity        | rotate         | mlp_1hidden |           12000 |          3000 |          10000 |       0.938    |       0.937405 |          0.7101 |        0.707363 |      0.841977 |
| required_same_corruption | identity        | identity       | mlp_2hidden |           12000 |          3000 |          10000 |       0.945667 |       0.945139 |          0.9486 |        0.947998 |      1.62552  |
| identity_to_noise        | identity        | shot_noise     | mlp_2hidden |           12000 |          3000 |          10000 |       0.945667 |       0.945139 |          0.9308 |        0.929783 |      1.62552  |
| identity_to_noise        | identity        | rotate         | mlp_2hidden |           12000 |          3000 |          10000 |       0.945667 |       0.945139 |          0.7296 |        0.727112 |      1.62552  |
| required_same_corruption | identity        | identity       | pca_mlp     |           12000 |          3000 |          10000 |       0.904    |       0.902969 |          0.9103 |        0.908991 |      0.777249 |
| identity_to_noise        | identity        | shot_noise     | pca_mlp     |           12000 |          3000 |          10000 |       0.904    |       0.902969 |          0.8945 |        0.892804 |      0.777249 |
| identity_to_noise        | identity        | rotate         | pca_mlp     |           12000 |          3000 |          10000 |       0.904    |       0.902969 |          0.6291 |        0.62701  |      0.777249 |
| required_same_corruption | shot_noise      | shot_noise     | mlp_1hidden |           12000 |          3000 |          10000 |       0.919    |       0.918033 |          0.923  |        0.922043 |      0.868158 |
| required_same_corruption | shot_noise      | shot_noise     | mlp_2hidden |           12000 |          3000 |          10000 |       0.925333 |       0.924574 |          0.9319 |        0.931018 |      1.40945  |
| required_same_corruption | shot_noise      | shot_noise     | pca_mlp     |           12000 |          3000 |          10000 |       0.891667 |       0.890242 |          0.8986 |        0.897174 |      0.778439 |
| required_same_corruption | rotate          | rotate         | mlp_1hidden |           12000 |          3000 |          10000 |       0.924667 |       0.923852 |          0.9229 |        0.921902 |      0.856548 |
| required_same_corruption | rotate          | rotate         | mlp_2hidden |           12000 |          3000 |          10000 |       0.929333 |       0.928521 |          0.9291 |        0.928391 |      1.42188  |
| required_same_corruption | rotate          | rotate         | pca_mlp     |           12000 |          3000 |          10000 |       0.863333 |       0.861166 |          0.8682 |        0.865709 |      0.696613 |

![必做同分布测试准确率](outputs/figures/required_same_corruption_accuracy.png)

![identity 训练模型跨噪声测试准确率](outputs/figures/identity_to_noise_accuracy.png)

![最佳必做模型混淆矩阵](outputs/figures/best_required_confusion_matrix.png)

必做同分布测试中，最佳组合是 `identity` 上的 `mlp_2hidden`，测试准确率为 0.949。从 identity 跨到噪声测试集时，性能通常明显下降，说明普通 MNIST 笔画特征对分布偏移不够稳健。`shot_noise` 主要破坏局部像素，`rotate` 则改变几何形态，二者造成的错误类型不同。

## 11. 选做部分实验结果

选做部分将 `identity`、`shot_noise`、`rotate` 的训练样本合并，训练一个双隐层 MLP，再在本地全部 16 个 MNIST-C 测试集上评价。

| experiment              | train_datasets             | eval_dataset   | model       |   train_samples |   val_samples |   test_samples |   val_accuracy |   val_macro_f1 |   test_accuracy |   test_macro_f1 |   fit_seconds |
|:------------------------|:---------------------------|:---------------|:------------|----------------:|--------------:|---------------:|---------------:|---------------:|----------------:|----------------:|--------------:|
| optional_combined_train | identity+shot_noise+rotate | identity       | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.9516 |       0.951012  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | shot_noise     | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.932  |       0.931043  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | impulse_noise  | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.1091 |       0.0883086 |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | glass_blur     | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.8026 |       0.803353  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | motion_blur    | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.3815 |       0.374047  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | shear          | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.6683 |       0.672878  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | scale          | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.3649 |       0.299449  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | rotate         | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.9193 |       0.918226  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | brightness     | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.101  |       0.0183503 |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | translate      | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.161  |       0.146349  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | stripe         | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.101  |       0.018347  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | fog            | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.1014 |       0.0209488 |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | spatter        | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.1469 |       0.134378  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | dotted_line    | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.4323 |       0.438714  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | zigzag         | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.3484 |       0.353495  |       2.25557 |
| optional_combined_train | identity+shot_noise+rotate | canny_edges    | mlp_2hidden |           19200 |          4800 |          10000 |       0.947083 |       0.946574 |          0.627  |       0.632138  |       2.25557 |

![选做组合模型在全部测试集上的准确率](outputs/figures/optional_all_datasets_accuracy.png)

组合训练模型表现最好的测试集是 `identity`，准确率为 0.952；表现最差的是 `brightness`，准确率为 0.101。训练集中包含的噪声类型通常更容易被识别；未见过的几何变换、边缘化或强遮挡类破坏更容易导致准确率下降。

## 12. 样例与误分类可视化

下图展示本地全部 MNIST-C 子集的样例，便于直观比较不同破坏类型。误分类图来自选做组合模型表现最差的测试集，标题中的 `T` 表示真实标签，`P` 表示预测标签。

![MNIST-C 各子集样例](outputs/figures/mnist_c_dataset_examples.png)

![选做模型误分类样例](outputs/figures/optional_worst_misclassified_examples.png)

样例图说明：随机噪声、线条遮挡、亮度变化、边缘化等破坏会改变像素分布，而 MLP 没有卷积结构中的局部平移不变性，因此在未见过或视觉形态变化较大的数据集上更容易误判。

## 13. 测试代码

下面的代码可以重新运行全部实验并刷新结果文件。默认参数使用分层抽样以便在普通电脑上完成；若希望使用完整训练集，可把 `--train-limit` 和 `--optional-train-limit` 设置为 `0`。

In [ ]:
# 重新运行实验：
# !python main.py --train-limit 15000 --optional-train-limit 8000 --max-iter 12

# 使用完整训练集：
# !python main.py --train-limit 0 --optional-train-limit 0 --max-iter 20

## 14. 结论

1. 数据处理方面，标准化对 MLP 训练最重要，PCA 可作为速度和鲁棒性的折中，但不一定提升最高准确率。
2. 模型方面，双隐层 MLP 通常比单隐层模型有更强表示能力，PCA+MLP 在部分噪声上更稳定但上限较低。
3. 损失函数方面，交叉熵更适合多分类概率学习；标签平滑适合作为增强鲁棒性的候选策略；MSE 的分类优化效率较低。
4. 无监督学习方面，KMeans 能给出可解释聚类基线，但缺少类别边界学习，准确率明显低于监督神经网络。
5. 验证方法方面，3 折交叉验证比单次 hold-out 更稳定，适合用于说明模型选择的可靠性。
6. 鲁棒性方面，identity 上训练的模型跨到噪声测试集时明显下降，说明多噪声训练或数据增强是提升分布外鲁棒性的关键。